## Faiss

Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings

In [2]:
loader=TextLoader('speech.txt')
documents=loader.load()

text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs=text_splitter.split_documents(documents)

In [4]:
embeddings=OllamaEmbeddings(model="nomic-embed-text")

db=FAISS.from_documents(docs,embeddings)
db

In [6]:
#### querying

query="How does the speaker describe the desired output of the war?"
docs=db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

## As a Retriever

We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [8]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

## Similarity Search with score

There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [9]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='48eed47b-a3af-434d-a246-096fe4d0128a', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(371.59317)),
 (Document(id='7ea396ba-d0c1-4c72-87da-1a3d5bdc6b3f', metadata={'source': 'spe

In [10]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[-0.539386510848999,
 0.6969547271728516,
 -3.98321270942688,
 -0.8051780462265015,
 1.8211506605148315,
 0.9838442206382751,
 -0.631679117679596,
 0.11198727041482925,
 0.30263248085975647,
 0.20640215277671814,
 0.34659284353256226,
 0.7697103023529053,
 1.205575942993164,
 1.4715697765350342,
 2.520134687423706,
 -0.576102614402771,
 -0.4480063021183014,
 -1.2722896337509155,
 -0.6009804606437683,
 0.7067451477050781,
 -0.3098032772541046,
 -0.6507505178451538,
 0.2669380307197571,
 -0.03229525685310364,
 1.1336288452148438,
 0.500370979309082,
 -0.6335601806640625,
 0.06341487169265747,
 -0.7360973954200745,
 -0.5294520854949951,
 1.0159146785736084,
 0.018135875463485718,
 -0.340323805809021,
 0.6192439794540405,
 -1.6993653774261475,
 -1.7529014348983765,
 -0.02596450224518776,
 0.8939011096954346,
 0.43456265330314636,
 -1.0393884181976318,
 0.012287267483770847,
 -0.6819247007369995,
 -0.01702888496220112,
 -1.1439595222473145,
 0.7021575570106506,
 -0.14845292270183563,
 -0.13

In [11]:
docs_score=db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='48eed47b-a3af-434d-a246-096fe4d0128a', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='7ea396ba-d0c1-4c72-87da-1a3d5bdc6b3f', metadata={'source': 'speech.txt'}, page_content='The

## Saving and Loading

In [12]:
db.save_local("faiss_index")

In [15]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)

In [16]:
docs

[Document(id='48eed47b-a3af-434d-a246-096fe4d0128a', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='7ea396ba-d0c1-4c72-87da-1a3d5bdc6b3f', metadata={'source': 'speech.txt'}, page_content='The